# DPO Alignment Notebook
## Stage 3: Direct Preference Optimization

This notebook performs DPO alignment using preference data to improve answer quality.

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes accelerate unsloth[colab-new] trl -U

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer
import json
import os

print("=" * 60)
print("STAGE 3: DPO ALIGNMENT")
print("=" * 60)

print(f\"GPU Available: {torch.cuda.is_available()}\")
if torch.cuda.is_available():
    print(f\"GPU Name: {torch.cuda.get_device_name(0)}\\n")

## Step 1: Load Preference Dataset

In [ ]:
print("[STEP 1] Loading preference dataset...")
pref_dataset_path = 'course-doubt-assistant/data/preference_dataset.jsonl'

data = []
with open(pref_dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f\"✓ Loaded {len(data)} preference examples\")
print(f\"\\nFirst example:\")
print(f\"  Prompt: {data[0]['prompt']}\")
print(f\"  Chosen: {data[0]['chosen'][:80]}...\")
print(f\"  Rejected: {data[0]['rejected'][:80]}...\\n")

## Step 2: Create Dataset Object

In [ ]:
print("[STEP 2] Creating dataset object...")
dataset = Dataset.from_dict({
    'prompt': [d['prompt'] for d in data],
    'chosen': [d['chosen'] for d in data],
    'rejected': [d['rejected'] for d in data]
})

print(f\"✓ Dataset size: {len(dataset)}\")
print(f\"✓ Columns: {dataset.column_names}\\n")

## Step 3: Load Model and Tokenizer

In [ ]:
print("[STEP 3] Loading model and tokenizer...")
MODEL_NAME = 'unsloth/tinyllama-bnb-4bit'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)

print(f\"✓ Model loaded: {MODEL_NAME}\\n")

## Step 4: Configure LoRA for DPO

In [ ]:
print("[STEP 4] Configuring LoRA for DPO...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
    modules_to_save=['lm_head']
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f\"✓ Trainable params: {trainable_params:,}\")
print(f\"✓ Trainable %: {100 * trainable_params / total_params:.2f}%\\n")

## Step 5: Configure DPO Training

In [ ]:
print("[STEP 5] Configuring DPO training arguments...")
os.makedirs('./outputs/dpo_aligned', exist_ok=True)

training_args = TrainingArguments(
    output_dir='./outputs/dpo_aligned',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=5,
    save_steps=25,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    optim='adamw_8bit',
    seed=42,
    report_to=[],
    max_steps=500
)

print("✓ DPO training arguments configured\\n")

## Step 6: Initialize DPO Trainer

In [ ]:
print("[STEP 6] Initializing DPO Trainer...")

dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    beta=0.1,
    loss_type='sigmoid'
)

print("✓ DPO Trainer initialized\\n")

## Step 7: Train with DPO

In [ ]:
print("[STEP 7] Starting DPO alignment training...")
print("-" * 60)

train_result = dpo_trainer.train()
print("-" * 60)
print(f\"✓ DPO training loss: {train_result.training_loss:.4f}\\n")

## Step 8: Save DPO-Aligned Model

In [ ]:
print("[STEP 8] Saving DPO-aligned adapter...")
os.makedirs('./models/dpo_aligned_adapter', exist_ok=True)

dpo_adapter_path = './models/dpo_aligned_adapter'
model.save_pretrained(dpo_adapter_path)
tokenizer.save_pretrained(dpo_adapter_path)

print(f\"✓ DPO adapter saved to {dpo_adapter_path}\\n")

## Step 9: Test DPO-Aligned Model

In [ ]:
print("[STEP 9] Testing DPO-aligned model...")
print("-" * 60)

def generate_response(question, max_length=200):
    prompt = f\"### Instruction:
{question}
### Response:
\"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.5,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in response:
        response = response.split("### Response:")[1].strip()
    return response

test_questions = [
    'What is supervised learning?',
    'How does backpropagation work?',
    'Explain cross-validation'
]

for q in test_questions:
    print(f\"\\nQuestion: {q}\")
    answer = generate_response(q)
    print(f\"Answer: {answer[:200]}...\")

print("\\n" + "=" * 60)
print("✓ STAGE 3 COMPLETE!")
print("=" * 60)
print("\\n🎉 ALL THREE STAGES COMPLETE!")
print("Your model is ready for inference!")